In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score, recall_score, classification_report
from sklearn.pipeline import Pipeline
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
import re
import joblib

In [3]:
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
# Preprocessing function
def preprocess_text(text):
    try:
        # Convert to lowercase
        text = text.lower()
        # Remove special characters and digits
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        # Tokenize
        tokens = word_tokenize(text)
        # Remove stopwords
        stop_words = set(stopwords.words('english'))
        tokens = [token for token in tokens if token not in stop_words]
        # Join tokens back to string
        return ' '.join(tokens)
    except Exception as e:
        print(f"Error in preprocessing: {e}")
        return text


In [5]:
#map indices to emotions 
emotions = {
        0: 'sadness',
        1: 'joy',
        2: 'love',
        3: 'anger',
        4: 'fear',
        5: 'surprise'
}

# Load dataset (placeholder for Emotion Dataset)
def load_dataset():
    data = pd.read_csv("emotions.csv")
    data.columns
    data["processed_text"] = data["text"].apply(preprocess_text)
    return data


In [6]:
# Main function for emotion detection
def train_emotion_classifier():
    # Load and preprocess data
    df = load_dataset()
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        df['processed_text'], df['label'], test_size=0.2, random_state=42
    )
    
    # Create pipeline with TF-IDF and Logistic Regression
    clf = Pipeline([
        ('vectorizer', TfidfVectorizer(max_features=5000)),
        ('classifier', LogisticRegression(max_iter=1000))
    ])
    
    # Train the model
    clf.fit(X_train, y_train)
    
    # Evaluate the model
    y_pred = clf.predict(X_test)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    
    print("Model Evaluation Metrics:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print("\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred))
    
    return clf


In [7]:
# Function to predict emotion on new text
def predict_emotion(text, model):
    processed_text = preprocess_text(text)
    prediction = model.predict([processed_text])[0]
    return prediction


In [8]:

# Train the model
model = train_emotion_classifier()
    
# Save the model for integration
joblib.dump(model, 'emotion_classifier.pkl')
    
# Example predictions
test_texts = [
        "i was beaten by a dog",
        "Feeling so down after the news."
]
print("\nExample Text Predictions:")
for text in test_texts:
    emotion = predict_emotion(text, model)
    print(f"Text: {text}")
    print(f"Predicted Emotion: {emotions[emotion]}\n")

Model Evaluation Metrics:
Accuracy: 0.9009
Precision: 0.8999
Recall: 0.9009

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.95      0.94     24201
           1       0.92      0.93      0.92     28164
           2       0.81      0.77      0.79      6929
           3       0.91      0.90      0.91     11441
           4       0.85      0.85      0.85      9594
           5       0.78      0.71      0.74      3033

    accuracy                           0.90     83362
   macro avg       0.87      0.85      0.86     83362
weighted avg       0.90      0.90      0.90     83362


Example Text Predictions:
Text: i was beaten by a dog
Predicted Emotion: sadness

Text: Feeling so down after the news.
Predicted Emotion: joy

